<a href="https://colab.research.google.com/github/navibzy/ml-robustness-mnist/blob/main/01_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, random
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

device = "cuda" if torch.cuda.is_available() else "cpu"

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

tf = transforms.ToTensor()   # pixels stay in [0,1], no normalization
train_full = datasets.MNIST("data", train=True, download=True, transform=tf)
test_set   = datasets.MNIST("data", train=False, download=True, transform=tf)

# fixed split: 55,000 train / 5,000 validation
train_set, val_set = random_split(
    train_full, [55000, 5000], generator=torch.Generator().manual_seed(0))

train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_set, batch_size=1000)
test_loader  = DataLoader(test_set, batch_size=1000)

100%|██████████| 9.91M/9.91M [00:00<00:00, 20.4MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 492kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.53MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 3.83MB/s]


In [2]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.c1 = nn.Conv2d(1, 32, 3)
        self.c2 = nn.Conv2d(32, 64, 3)
        self.f1 = nn.Linear(64*5*5, 128)
        self.f2 = nn.Linear(128, 10)
    def forward(self, x):
        x = F.max_pool2d(F.relu(self.c1(x)), 2)   # 28 -> 26 -> 13
        x = F.max_pool2d(F.relu(self.c2(x)), 2)   # 13 -> 11 -> 5
        x = x.flatten(1)
        return self.f2(F.relu(self.f1(x)))

In [3]:
def train(model, loader, epochs=10, lr=0.01):
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    for ep in range(epochs):
        model.train()
        total = 0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            loss = F.cross_entropy(model(x), y)
            loss.backward(); opt.step()
            total += loss.item()
        print(f"epoch {ep+1}: train loss {total/len(loader):.4f}")
    return model

@torch.no_grad()
def accuracy(model, loader):
    model.eval()
    correct = n = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        correct += (model(x).argmax(1) == y).sum().item()
        n += len(y)
    return correct / n

In [4]:
set_seed(0)
model = CNN().to(device)
train(model, train_loader, epochs=10)
print("val acc:", accuracy(model, val_loader))
print("test acc:", accuracy(model, test_loader))
torch.save(model.state_dict(), "baseline_seed0.pt")

epoch 1: train loss 0.3579
epoch 2: train loss 0.0747
epoch 3: train loss 0.0532
epoch 4: train loss 0.0403
epoch 5: train loss 0.0323
epoch 6: train loss 0.0275
epoch 7: train loss 0.0223
epoch 8: train loss 0.0199
epoch 9: train loss 0.0168
epoch 10: train loss 0.0131
val acc: 0.9874
test acc: 0.9913
